# Evidence Timeline Reconstruction — EDA & Visualization
**UCF-Crime Dataset | 14 Classes | Deep Learning Pipeline**

This notebook covers:
1. Dataset distribution analysis
2. Class-wise frame visualization
3. Augmented image visualization
4. Optical flow visualization
5. Segmentation overlay visualization
6. Pixel intensity histograms (class-wise)
7. Training metrics comparison
8. Confusion matrix heatmaps
9. GradCAM / XAI visualization
10. Model leaderboard table

In [ ]:
import os, sys, json, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
from pathlib import Path
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.insert(0, str(Path('..').resolve()))

# Load config
with open('../configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

CLASSES = cfg['dataset']['classes']
FRAMES_DIR = Path('../data/processed/frames')
FLOW_DIR   = Path('../data/processed/optical_flow')
SEG_DIR    = Path('../data/processed/segments')
SPLITS_DIR = Path('../data/splits')
RESULTS_DIR = Path('../results')

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print('Setup complete. Classes:', CLASSES)

## 1. Dataset Distribution

In [ ]:
splits_path = SPLITS_DIR / 'splits.json'
assert splits_path.exists(), 'Run preprocess.py first!'

with open(splits_path) as f:
    splits = json.load(f)

# Count videos per class per split
from collections import Counter
records = []
for split, entries in splits.items():
    counts = Counter(e['class'] for e in entries)
    for cls in CLASSES:
        records.append({'split': split, 'class': cls, 'count': counts.get(cls, 0)})

df = pd.DataFrame(records)
total_per_class = df.groupby('class')['count'].sum().reset_index()
total_per_class.columns = ['class', 'total']
print(total_per_class.sort_values('total', ascending=False).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Stacked bar: train/val/test per class
pivot = df.pivot(index='class', columns='split', values='count').fillna(0)
pivot = pivot.reindex(CLASSES)
colors = {'train': '#3498db', 'val': '#e67e22', 'test': '#2ecc71'}
bottom = np.zeros(len(pivot))
for sp in ['train', 'val', 'test']:
    if sp in pivot.columns:
        axes[0].bar(pivot.index, pivot[sp], bottom=bottom,
                    label=sp.capitalize(), color=colors[sp], alpha=0.85)
        bottom += pivot[sp].values
axes[0].set_title('Video Distribution by Class & Split', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Number of Videos')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3, axis='y')

# Pie chart: total split distribution
split_totals = df.groupby('split')['count'].sum()
axes[1].pie(split_totals.values, labels=split_totals.index,
            autopct='%1.1f%%', colors=['#3498db','#e67e22','#2ecc71'],
            startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Train / Val / Test Split', fontsize=13, fontweight='bold')

plt.suptitle('UCF-Crime Dataset Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'plots/eda_distribution.png', bbox_inches='tight')
plt.show()

## 2. Class-wise Frame Visualization

In [ ]:
def get_sample_frame(cls_name, split='train', frame_idx='middle'):
    """Get a sample frame for a given class."""
    entries = [e for e in splits[split] if e['class'] == cls_name]
    if not entries:
        return None
    entry = entries[0]
    frames = sorted(Path(entry['frame_dir']).glob('*.jpg'))
    if not frames:
        return None
    idx = len(frames) // 2 if frame_idx == 'middle' else 0
    return cv2.cvtColor(cv2.imread(str(frames[idx])), cv2.COLOR_BGR2RGB)

# Grid: one frame per class
n_cols = 4
n_rows = int(np.ceil(len(CLASSES) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 5))
axes = axes.flatten()

for i, cls in enumerate(CLASSES):
    img = get_sample_frame(cls)
    if img is not None:
        axes[i].imshow(img)
        axes[i].set_title(cls, fontsize=11, fontweight='bold', color='#2c3e50')
    else:
        axes[i].text(0.5, 0.5, 'No frames\n(run preprocess.py)',
                     ha='center', va='center', color='gray', fontsize=9)
        axes[i].set_facecolor('#f0f0f0')
        axes[i].set_title(cls, fontsize=11, color='gray')
    axes[i].axis('off')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('UCF-Crime — Sample Frames by Class', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'plots/eda_classwise_frames.png', bbox_inches='tight')
plt.show()

## 3. Augmented Image Visualization

In [ ]:
import torch
from src.data.dataset import get_transforms, get_advanced_augmentation

def show_augmentations(cls_name='Assault', n_aug=8):
    entry = next((e for e in splits['train'] if e['class'] == cls_name), None)
    if not entry:
        print(f'No train entry for {cls_name}')
        return
    frames = sorted(Path(entry['frame_dir']).glob('*.jpg'))
    if not frames:
        print('No frames found. Run preprocess.py first.')
        return

    orig_img = Image.open(frames[len(frames)//2]).convert('RGB')
    adv_transform = get_advanced_augmentation(224)

    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])

    def to_display(t):
        img = t.numpy().transpose(1,2,0)
        img = std * img + mean
        return np.clip(img, 0, 1)

    fig, axes = plt.subplots(2, (n_aug + 2) // 2, figsize=(20, 8))
    axes = axes.flatten()
    axes[0].imshow(orig_img)
    axes[0].set_title('Original', fontsize=10, fontweight='bold')
    axes[0].axis('off')

    for i in range(1, n_aug + 1):
        aug = adv_transform(orig_img)
        axes[i].imshow(to_display(aug))
        axes[i].set_title(f'Aug #{i}', fontsize=9)
        axes[i].axis('off')

    for j in range(n_aug + 1, len(axes)):
        axes[j].axis('off')

    plt.suptitle(f'Advanced Augmentation — Class: {cls_name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'plots/eda_augmentation_{cls_name}.png', bbox_inches='tight')
    plt.show()

show_augmentations('Assault', n_aug=7)

## 4. Optical Flow Visualization

In [ ]:
def visualize_optical_flow(cls_name='Fighting', n_frames=6):
    entry = next((e for e in splits['train'] if e['class'] == cls_name), None)
    if not entry:
        print(f'No entry for {cls_name}')
        return

    rgb_frames   = sorted(Path(entry['frame_dir']).glob('*.jpg'))
    flow_frames  = sorted(Path(entry['flow_dir']).glob('*.png'))

    if not flow_frames:
        print('No optical flow frames. Run preprocess.py without --skip-flow')
        return

    step = max(1, len(rgb_frames) // n_frames)
    sel_rgb  = rgb_frames[::step][:n_frames]
    sel_flow = flow_frames[::step][:n_frames]

    fig, axes = plt.subplots(2, n_frames, figsize=(n_frames * 3.5, 7))
    for i, (rf, ff) in enumerate(zip(sel_rgb, sel_flow)):
        rgb_img  = cv2.cvtColor(cv2.imread(str(rf)), cv2.COLOR_BGR2RGB)
        flow_img = cv2.cvtColor(cv2.imread(str(ff)), cv2.COLOR_BGR2RGB)
        axes[0, i].imshow(rgb_img);  axes[0, i].axis('off')
        axes[1, i].imshow(flow_img); axes[1, i].axis('off')
        if i == 0:
            axes[0, i].set_ylabel('RGB Frames', fontsize=10)
            axes[1, i].set_ylabel('Optical Flow', fontsize=10)

    plt.suptitle(f'Optical Flow — Class: {cls_name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'plots/eda_optflow_{cls_name}.png', bbox_inches='tight')
    plt.show()

visualize_optical_flow('Fighting')

## 5. Segmentation Visualization

In [ ]:
def visualize_segmentation(cls_name='Robbery', n_samples=4):
    entry = next((e for e in splits['train'] if e['class'] == cls_name), None)
    if not entry:
        print(f'No entry for {cls_name}')
        return

    vid_stem  = entry['video_id']
    seg_video_dir = SEG_DIR / cls_name / vid_stem
    rgb_dir   = Path(entry['frame_dir'])

    segs  = sorted(seg_video_dir.glob('seg_*.png'))
    masks = sorted(seg_video_dir.glob('mask_*.png'))

    if not segs:
        print('No segmentation outputs. Run src/data/segmentation.py first.')
        return

    step = max(1, len(segs) // n_samples)
    sel_segs  = segs[::step][:n_samples]
    sel_masks = masks[::step][:n_samples] if masks else [None] * n_samples

    fig, axes = plt.subplots(3, n_samples, figsize=(n_samples * 4, 11))
    for i, (sf, mf) in enumerate(zip(sel_segs, sel_masks)):
        idx_str = sf.stem.split('_')[-1]
        rgb_path = rgb_dir / f'frame_{idx_str}.jpg'

        if rgb_path.exists():
            rgb = cv2.cvtColor(cv2.imread(str(rgb_path)), cv2.COLOR_BGR2RGB)
            axes[0, i].imshow(rgb)
            axes[0, i].set_title(f'Original #{idx_str}', fontsize=9)
        else:
            axes[0, i].axis('off')
        axes[0, i].axis('off')

        seg_img = cv2.cvtColor(cv2.imread(str(sf)), cv2.COLOR_BGR2RGB)
        axes[1, i].imshow(seg_img)
        axes[1, i].set_title(f'Segment Overlay', fontsize=9)
        axes[1, i].axis('off')

        if mf and Path(mf).exists():
            mask_img = cv2.imread(str(mf), cv2.IMREAD_GRAYSCALE)
            axes[2, i].imshow(mask_img, cmap='gray')
            axes[2, i].set_title(f'Binary Mask', fontsize=9)
        else:
            axes[2, i].axis('off')
        axes[2, i].axis('off')

    plt.suptitle(f'Segmentation Results — Class: {cls_name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'plots/eda_segmentation_{cls_name}.png', bbox_inches='tight')
    plt.show()

visualize_segmentation('Robbery')

## 6. Pixel Intensity Histograms (Class-wise)

In [ ]:
def compute_class_histogram(cls_name, n_videos=3, n_frames=10):
    """Compute mean RGB histogram across videos in a class."""
    entries = [e for e in splits['train'] if e['class'] == cls_name][:n_videos]
    hists = {c: np.zeros(64) for c in range(3)}
    count = 0
    for entry in entries:
        frames = sorted(Path(entry['frame_dir']).glob('*.jpg'))[:n_frames]
        for fp in frames:
            img = cv2.imread(str(fp))
            if img is None: continue
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            for ch in range(3):
                h = cv2.calcHist([img_rgb], [ch], None, [64], [0, 256]).flatten()
                hists[ch] += h
            count += 1
    if count > 0:
        for ch in hists:
            hists[ch] /= (hists[ch].sum() + 1e-8)
    return hists

# Compare anomaly vs normal
compare_classes = ['Normal', 'Fighting', 'Robbery', 'Explosion']
ch_colors = ['red', 'green', 'blue']
ch_names  = ['Red', 'Green', 'Blue']

fig, axes = plt.subplots(len(compare_classes), 3, figsize=(16, len(compare_classes)*3))
x = np.arange(64)

for i, cls in enumerate(compare_classes):
    hists = compute_class_histogram(cls)
    for ch in range(3):
        axes[i, ch].fill_between(x, hists[ch], alpha=0.6, color=ch_colors[ch])
        axes[i, ch].plot(x, hists[ch], color=ch_colors[ch], linewidth=1.2)
        if ch == 0:
            axes[i, ch].set_ylabel(cls, fontsize=10, fontweight='bold', rotation=90)
        if i == 0:
            axes[i, ch].set_title(f'{ch_names[ch]} Channel', fontsize=10)
        axes[i, ch].set_xlim(0, 63)
        axes[i, ch].set_ylim(0, None)
        axes[i, ch].grid(True, alpha=0.3)
        axes[i, ch].tick_params(labelsize=7)

plt.suptitle('Class-wise Pixel Intensity Histograms', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'plots/eda_histograms_classwise.png', bbox_inches='tight')
plt.show()

## 7. Training Metrics — Learning Curves Comparison

In [ ]:
logs_dir = RESULTS_DIR / 'logs'
log_files = list(logs_dir.glob('*_history.json'))

if not log_files:
    print('No training history found. Run train.py first.')
else:
    histories = {}
    for lf in log_files:
        name = lf.stem.replace('_history', '')
        with open(lf) as f:
            histories[name] = json.load(f)

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    cmap = plt.cm.tab10

    for i, (name, h) in enumerate(histories.items()):
        color = cmap(i / max(1, len(histories)))
        ep = range(1, len(h['train_loss']) + 1)
        axes[0].plot(ep, h['val_loss'],  color=color, label=name, linewidth=1.5)
        axes[1].plot(ep, h['val_acc'],   color=color, label=name, linewidth=1.5)
        axes[2].plot(ep, h['val_f1'],    color=color, label=name, linewidth=1.5)

    for ax, title, ylabel in zip(
        axes,
        ['Validation Loss', 'Validation Accuracy', 'Validation F1 Macro'],
        ['Loss', 'Accuracy', 'F1 Score']
    ):
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
        ax.legend(fontsize=7, loc='best')
        ax.grid(True, alpha=0.3)

    plt.suptitle('Learning Curves — All Models', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'plots/eda_all_learning_curves.png', bbox_inches='tight')
    plt.show()

## 8. Model Leaderboard & Metrics Comparison

In [ ]:
metrics_path = RESULTS_DIR / 'metrics' / 'all_models_metrics.json'

if not metrics_path.exists():
    print('No metrics found. Run evaluate.py first.')
else:
    with open(metrics_path) as f:
        all_metrics = json.load(f)

    rows = []
    for name, m in all_metrics.items():
        rows.append({
            'Model': name,
            'Accuracy':    round(m.get('accuracy', 0), 4),
            'Precision':   round(m.get('precision', 0), 4),
            'Recall':      round(m.get('recall', 0), 4),
            'F1 Macro':    round(m.get('f1_macro', 0), 4),
            'F1 Weighted': round(m.get('f1_weighted', 0), 4),
            'ROC-AUC':     round(m.get('roc_auc') or 0, 4),
        })

    leaderboard = pd.DataFrame(rows).sort_values('F1 Macro', ascending=False)
    leaderboard.index = range(1, len(leaderboard)+1)
    leaderboard.index.name = 'Rank'

    print('\n=== MODEL LEADERBOARD ===')
    print(leaderboard.to_string())

    # Heatmap
    metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1 Macro', 'F1 Weighted', 'ROC-AUC']
    fig, ax = plt.subplots(figsize=(14, max(4, len(rows) * 0.6 + 2)))
    heat_data = leaderboard[metric_cols].astype(float)
    sns.heatmap(heat_data, annot=True, fmt='.3f', cmap='YlOrRd',
                xticklabels=metric_cols,
                yticklabels=leaderboard['Model'].values,
                ax=ax, linewidths=0.5, cbar_kws={'label': 'Score'})
    ax.set_title('Model Performance Heatmap', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'plots/eda_model_heatmap.png', bbox_inches='tight')
    plt.show()

## 9. Confusion Matrix — Best Model

In [ ]:
if metrics_path.exists():
    best_model = leaderboard.iloc[0]['Model']
    indiv_path = RESULTS_DIR / 'metrics' / f'{best_model}_metrics.json'

    if indiv_path.exists():
        with open(indiv_path) as f:
            bm = json.load(f)

        if 'classification_report' in bm:
            report = bm['classification_report']
            cls_rows = []
            for cls in CLASSES:
                if cls in report:
                    r = report[cls]
                    cls_rows.append({
                        'Class':     cls,
                        'Precision': round(r.get('precision', 0), 3),
                        'Recall':    round(r.get('recall', 0), 3),
                        'F1-Score':  round(r.get('f1-score', 0), 3),
                        'Support':   int(r.get('support', 0)),
                    })
            cls_df = pd.DataFrame(cls_rows)
            print(f'\nClassification Report — {best_model}')
            print(cls_df.to_string(index=False))

            # Class F1 bar chart
            fig, ax = plt.subplots(figsize=(14, 5))
            colors = plt.cm.RdYlGn(cls_df['F1-Score'].values)
            bars = ax.bar(cls_df['Class'], cls_df['F1-Score'], color=colors)
            ax.set_ylim(0, 1.05)
            ax.set_xlabel('Class'); ax.set_ylabel('F1-Score')
            ax.set_title(f'{best_model} — Class-wise F1 Score', fontsize=12, fontweight='bold')
            ax.tick_params(axis='x', rotation=45)
            ax.grid(True, alpha=0.3, axis='y')
            for bar, val in zip(bars, cls_df['F1-Score']):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                        f'{val:.3f}', ha='center', va='bottom', fontsize=8)
            plt.tight_layout()
            plt.savefig(RESULTS_DIR / f'plots/eda_classwise_f1_{best_model}.png', bbox_inches='tight')
            plt.show()
    else:
        print(f'No individual metrics file for {best_model}. Run evaluate.py.')
else:
    print('Run evaluate.py first.')

## 10. XAI — GradCAM Gallery

In [ ]:
xai_dir = RESULTS_DIR / 'xai' / 'gradcam'
gradcam_imgs = sorted(xai_dir.glob('*.png'))

if not gradcam_imgs:
    print('No GradCAM outputs. Run src/evaluation/xai.py first.')
else:
    show_n = min(12, len(gradcam_imgs))
    cols = 4
    rows = int(np.ceil(show_n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 4))
    axes = axes.flatten()

    for i, gp in enumerate(gradcam_imgs[:show_n]):
        img = cv2.cvtColor(cv2.imread(str(gp)), cv2.COLOR_BGR2RGB)
        axes[i].imshow(img)
        title = gp.stem.replace('gradcam_', '').replace('_', ' ')
        axes[i].set_title(title[:30], fontsize=8)
        axes[i].axis('off')

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.suptitle('GradCAM Explanations Gallery', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'plots/eda_gradcam_gallery.png', bbox_inches='tight')
    plt.show()
    print(f'Showing {show_n} of {len(gradcam_imgs)} GradCAM outputs')

---
## Summary
All plots saved to `results/plots/`. Use `results/evaluation_report.html` for the full report.